# Runbook de despliegue — Infraestructura chatbot UPC (AWS CDK)

Este notebook **orquesta y documenta** el despliegue de la arquitectura objetivo en AWS.
No *describe* la infra (eso lo hacen los stacks en `stacks/`); aqui solo se ejecuta el ciclo
**construir -> ver outputs -> destruir**, con la explicacion de cada recurso para la tesis.

> **Modelo de costos**: la cuenta es post-15-jul-2025 -> $200 en creditos (6 meses), NO horas gratis.
> Estrategia: desplegar en **rafagas** para la demo/sustentacion y hacer **tear-down** al terminar.

| Stack | Rol | Aporte al plan de continuidad |
|---|---|---|
| `ChatbotUpc-Network` | VPC, subnets, NAT, SGs | Aislamiento de red, 2 AZ |
| `ChatbotUpc-Data` | RDS pgvector, ElastiCache, S3, SQS | RDS Multi-AZ, Redis con replica, SQS durable |
| `ChatbotUpc-Auth` | Cognito | Login admin gestionado |
| `ChatbotUpc-Compute` | ALB, ASG (Next/FastAPI/Celery), IAM | Auto-reemplazo de instancias, balanceo multi-AZ |
| `ChatbotUpc-Observability` | CloudWatch | Deteccion de fallos (5xx, cola, RDS) |

## 0. Configuracion
Elige el perfil: `demo` (arquitectura objetivo, alta disponibilidad) o `frugal` (economico).

In [ ]:
import subprocess, os

MODE = "demo"            # "demo" | "frugal"
REGION = "us-east-1"
CDK_DIR = os.path.dirname(os.path.abspath("deploy_runbook.ipynb"))

def run(cmd):
    """Ejecuta un comando mostrando la salida en vivo."""
    print(f"$ {cmd}\n")
    p = subprocess.run(cmd, shell=True, cwd=CDK_DIR)
    if p.returncode != 0:
        raise RuntimeError(f"comando fallo (exit {p.returncode})")

print(f"Perfil: {MODE} | Region: {REGION}")

## 1. Verificar entorno y credenciales

In [ ]:
run("cdk --version")
run("aws sts get-caller-identity")

## 2. Bootstrap (solo la primera vez por cuenta/region)
Crea el bucket S3 y roles que CDK usa para subir los assets de los templates.

In [ ]:
run(f"cdk bootstrap aws://$(aws sts get-caller-identity --query Account --output text)/{REGION}")

## 3. Sintetizar y previsualizar cambios
`synth` genera el CloudFormation (entregable IaC); `diff` muestra que se va a crear.

In [ ]:
run(f"cdk synth --all -c mode={MODE}")
run(f"cdk diff --all -c mode={MODE}")

## 4. Desplegar  ⚠️ EMPIEZA EL GASTO DE CREDITOS
Despliega los 5 stacks en orden de dependencia automatico.
El perfil `demo` corriendo 24/7 quema ~$100-130/mes (los $200 duran ~6-8 semanas).

In [ ]:
# --require-approval never evita los prompts interactivos (ya revisamos el diff arriba)
run(f"cdk deploy --all -c mode={MODE} --require-approval never --outputs-file cdk.out/outputs-{MODE}.json")

## 5. Ver los outputs (URL del ALB, endpoints, etc.)

In [ ]:
import json
with open(os.path.join(CDK_DIR, f"cdk.out/outputs-{MODE}.json")) as f:
    outputs = json.load(f)
for stack, vals in outputs.items():
    print(f"\n=== {stack} ===")
    for k, v in vals.items():
        print(f"  {k}: {v}")

## 6. Habilitar pgvector en RDS (una vez)
El esquema ya define la extension (`infra/docker/postgres/init.sql`). Si la BD es nueva,
conectarse y ejecutar `CREATE EXTENSION IF NOT EXISTS vector;`. La contrasena esta en
Secrets Manager (`chatbot-upc/<mode>/db`).

In [ ]:
run(f"aws secretsmanager get-secret-value --secret-id chatbot-upc/{MODE}/db --query SecretString --output text --region {REGION}")

## 7. TEAR-DOWN — destruir todo y detener el gasto  🗑️
Borra los 5 stacks. Gracias a `RemovalPolicy.DESTROY` y `auto_delete_objects`,
RDS / S3 / Cognito se eliminan limpiamente. **Ejecutar al terminar la demo.**

In [ ]:
run(f"cdk destroy --all -c mode={MODE} --force")
print("\n🗑️  Infra destruida — gasto detenido")